In [0]:
spark.sql("USE CATALOG 03_gold")
spark.sql("USE SCHEMA default")

**All the fact and dim tables**

In [0]:
spark.sql("SHOW TABLES").show()

**TOTAL SALES: Total money collected from customers for all products sold.**

In [0]:
spark.sql("""
SELECT total_sales
FROM 03_gold.default.sales_cube
WHERE date IS NULL AND store_id IS NULL AND product_id IS NULL
""").show()

**PROFIT PER PRODUCT: Selling price minus the cost of the item.**

In [0]:
spark.sql("""
SELECT product_id, SUM(total_profit) product_profit
FROM 03_gold.default.fact_sales
GROUP BY product_id
""").show()

**TOP SELLING CATEGORY: Which group (e.g. Electronics) brings the most money.**

In [0]:
spark.sql("""
SELECT p.category, SUM(f.total_sales) AS revenue
FROM fact_sales f
JOIN dim_product p
ON f.product_id = p.product_id
GROUP BY p.category
ORDER BY revenue DESC 
LIMIT 1
""").show()

**BASKET SIZE: Average number of items bought per single transaction.**

In [0]:
spark.sql("""
SELECT AVG(quantity) AS avg_basket_size
FROM fact_sales
""").show()

**RETURN RATE: Percentage of total sales that were returned by customers.**

In [0]:
spark.sql("""
SELECT 
(COUNT(r.return_id) * 1.0 / COUNT(f.transaction_id)) * 100 AS return_rate
FROM 03_gold.default.fact_sales f
LEFT JOIN 02_silver.default.returns r
ON f.transaction_id = r.transaction_id
""").show()

**OUT-OF-STOCK COUNT: Count of unique products with zero units in stock.**

In [0]:
spark.sql("""
SELECT COUNT(DISTINCT product_id) AS out_of_stock
FROM 02_silver.default.inventory
WHERE stock_qty = 0
""").show()

**STORE PERFORMANCE: Ranking of the 5 stores based on daily sales volume.**

In [0]:
spark.sql("""
SELECT 
    store_id,
    date,
    SUM(total_sales) AS daily_sales
FROM 03_gold.default.fact_sales
GROUP BY store_id, date
ORDER BY daily_sales DESC
LIMIT 5
""").show()

**DISCOUNT IMPACT: Money lost due to differences in marked vs. sold price.**

In [0]:
spark.sql("""
SELECT 
    SUM(discount * quantity) AS discount_impact
FROM 03_gold.default.fact_sales
""").show()

**REPEAT CUSTOMER COUNT: Number of unique users with more than one purchase.**

In [0]:
spark.sql("""
SELECT COUNT(*) AS repeat_customers
FROM (
    SELECT customer_id
    FROM 03_gold.default.fact_sales
    WHERE customer_id != 'unknown'
    GROUP BY customer_id
    HAVING COUNT(DISTINCT transaction_id) > 1
)
""").show()

**SLOW-MOVING INVENTORY: Products with zero sales recorded in the last 30 days.**

In [0]:
spark.sql("""
SELECT p.product_id
FROM 03_gold.default.dim_product p
LEFT JOIN 03_gold.default.fact_sales f
ON p.product_id = f.product_id
AND f.date >= DATE_SUB(CURRENT_DATE(), 30)
WHERE f.product_id IS NULL
""").show()